# FinScope — Personal FP&A Platform (Google Colab)

This notebook runs the full project in Colab and pulls **real live market data** to drive the Monte Carlo planning model.

**Data sources**
- *Transactions*: synthetic (personal bank data is private — production apps use Plaid behind user consent; see the README for the Plaid path).
- *Market returns*: **live** S&P 500 history (yfinance, Stooq fallback).
- *Inflation*: **live** CPI from FRED (optional free key).

Run the cells top to bottom.

## 1. Clone the repo and install dependencies
Replace `YOUR_USERNAME` with your GitHub username once you've pushed the project.

In [ ]:
# If you've pushed FinScope to GitHub, clone it:
!git clone https://github.com/YOUR_USERNAME/finscope.git
%cd finscope
!pip install -q -r requirements.txt

## 2. Build & load the transaction data
Generate synthetic transactions, categorize them from their descriptions, and load everything into SQLite.

In [ ]:
import sys
sys.path.insert(0, 'src')   # make the finscope package importable

from finscope import data_generator, categorize, database

df = data_generator.generate_transactions(n_months=24)
cat = categorize.categorize(df)
print(f'Transactions: {len(df)}')
print(f'Categorization accuracy: {categorize.accuracy(cat):.1%}')

conn = database.connect()
database.init_db(conn)
database.load_transactions(conn, df)
database.load_budgets(conn)
actuals = database.monthly_actuals(conn)
actuals.tail()

## 3. Budget vs. Actual variance analysis
The core FP&A report: favorable (under budget) vs. unfavorable, in $ and %.

In [ ]:
import plotly.express as px
from finscope import variance

report = variance.variance_report(actuals)
summary = variance.variance_summary(report)
print(f"Total variance: ${summary['total_variance']:,.0f} ({summary['status']})")

fig = px.bar(report, x='category', y='variance', color='status',
             color_discrete_map={'Favorable': '#2e7d32', 'Unfavorable': '#c62828'},
             title='Variance by category (positive = under budget)')
fig.show()
report

## 4. Rolling cash-flow forecast + backtest
Exponential-smoothing forecast of net monthly cash flow, with a MAPE backtest on a holdout — the same accuracy check FP&A teams run on a rolling forecast.

In [ ]:
import plotly.graph_objects as go
from finscope import forecast

fc = forecast.forecast_cashflow(actuals, horizon=12)
mape = forecast.backtest_mape(actuals)
print(f'Backtest MAPE: {mape:.1%}')

fig = go.Figure()
for kind, color in (('actual', '#1565c0'), ('forecast', '#ef6c00')):
    sub = fc[fc['type'] == kind]
    fig.add_trace(go.Scatter(x=sub['month'], y=sub['actual'],
                             mode='lines+markers', name=kind, line=dict(color=color)))
fig.update_layout(title='Monthly net cash flow: actual vs. forecast')
fig.show()

## 5. LIVE market data → Monte Carlo planning
This is the real-data centerpiece. We pull **actual S&P 500 history**, estimate the real expected return and volatility, pull **live CPI inflation**, and feed all three into a 10,000-path net-worth simulation.

**Optional:** for live inflation, get a free FRED key at https://fred.stlouisfed.org/docs/api/api_key.html and paste it below. Without a key the model uses a default inflation rate; market returns are still live.

In [ ]:
import os, getpass

# Optional: paste your free FRED key for live inflation (press Enter to skip)
key = getpass.getpass('FRED API key (optional, Enter to skip): ').strip()
if key:
    os.environ['FRED_API_KEY'] = key

In [ ]:
from finscope import market_data, montecarlo

assumptions = market_data.derive_assumptions(
    ticker='^GSPC', start='2005-01-01',
    fred_api_key=os.environ.get('FRED_API_KEY'),
)
print('Live-derived assumptions:')
for k, v in assumptions.items():
    print(f'  {k}: {v}')

res = montecarlo.simulate(
    annual_return_mean=assumptions['annual_return_mean'],
    annual_return_std=assumptions['annual_return_std'],
    annual_inflation=assumptions['annual_inflation'],
)
print(f"\nProbability of reaching ${res.goal:,.0f} in {res.horizon_years} yrs: "
      f'{res.prob_goal:.1%}')
print({f'P{k}': f'${v:,.0f}' for k, v in res.percentiles().items()})

In [ ]:
fig = px.histogram(x=res.terminal, nbins=60,
                   title='Distribution of terminal net worth (real $)',
                   labels={'x': 'Net worth ($)'})
fig.add_vline(x=res.goal, line_dash='dash', line_color='red',
              annotation_text='Goal')
fig.show()

## 6. Personal financial statements & KPIs

In [ ]:
from finscope import statements

month = actuals['month'].max()
print('INCOME STATEMENT'); display(statements.income_statement(actuals, month))
print('BALANCE SHEET');    display(statements.balance_sheet())
print('KPIs');             print(statements.kpis(actuals, month=month))

## 7. (Optional) Run the full Streamlit dashboard from Colab
Colab can't show Streamlit inline, so we expose it through a tunnel. Run the cell, open the printed URL, and use the IP shown as the tunnel password.

In [ ]:
!npm install -q localtunnel
# write the sample DB the app reads
!python -m scripts.setup_data
!streamlit run app/streamlit_app.py &>/content/st_log.txt &
!echo 'Tunnel password (your public IP):' && curl -s ipv4.icanhazip.com
!npx localtunnel --port 8501